# 최종 테스트 및 제출

### 최종 학습 및 테스트
k-fold를 사용하지 않고 01에서 분리한 학습셋 전체를 사용하여 학습 후 test셋으로 모델을 평가한다.


In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score

train = pd.read_csv("../data/processed/01/train.csv")
test = pd.read_csv("../data/processed/01/test.csv")
model = joblib.load("../data/model/tuned_logistic_regression.pkl")
model.fit(train, train["Survived"])
predict = model.predict(test)

final_score = accuracy_score(predict, test["Survived"])
print(final_score)

0.7821229050279329


### 제출
마지막 raw데이터의 train.csv를 이용하여 최종 학습 후 test셋으로 예측 및 예측결과를 기록한다.

In [2]:
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
model.fit(train, train["Survived"])
predict = model.predict(test)

submission = pd.DataFrame(predict, columns=["Survived"])
submission["PassengerId"] = test["PassengerId"]
submission.to_csv("../data/submission/logistic_v7_tuned_full_train.csv", index=False)

## 기록
* 처음에 실수로 미세튜닝을 할 모델을 v5버전의 전처리기를 가져다가 썼다. 근데 gridsearch객체를 통한 k-fold 점수는 0.82231이 나왔다.
* 실수를 인지하고 학습셋 성능 중 가장 우수한 버전인 v7을 가져다가 학습시켰는데 미세튜닝이 끝나고 k-fold 최고 점수는 0.8189가 나왔다.

-> 따라서 예상할때는 v5버전의 캐글 LB점수가 더 우수할 것으로 생각하였으나 결과는 v5: 0.76076, v7: 0.77272가 나왔다.
* v5에서는 모든 특성을 범주화하여 정보를 단순화했음. v7는 나이와 부모 인원에 대해서 숫자 정보를 유지함
* 따라서 v7이 유지한 미세한 정보가 캐글의 실제 테스트 셋에서는 더 잘 작동했다고 해석할 수 있다.
* 실제 테스트 셋의 크기는 418밖에 안돼서 3~5명 더 맞춘 수준으로 점수가 훨씬 올라가기 때문에 단순 훈련 세트의 정보를 버리지 않고 유지하는 것만으로도 더 잘 작동함을 알 수 있다.
* 또한 CV1등이 실제 테스트 1등을 보장하지도 않고 오히려 CV테스트셋에 과적합될 가능성이 높음을 알 수 있음
* 단순화된 피처가 CV에서는 안정적이었어도 외부 데이터에는 정보 손실로 이어질 수 있음을 알 수 있다.